# Exploratory Data Analysis 6.0

**Goal:** evaluate whether rolling team-form features improve predictive performance on combined 2025 + 2026 match data.

## Hypothesis
Rolling features should become more useful with multi-year history because they reduce cold-start effects and better capture short-term momentum.

## Experiments
- Rolling-only model with a 5-game window (`wr_diff_5`)
- Combined model using all-time + rolling features
- Comparison against baseline all-time model accuracy (`62.63%`)

## 1) Import Libraries

Load data handling, model training, and evaluation utilities used throughout this notebook.

In [31]:
# Data manipulation
import pandas as pd

# Baseline model + split/evaluation utilities
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

## 2) Load and Chronologically Sort Combined Match Data

Read the combined processed table, parse dates, and sort by time so rolling features only use prior matches.

In [32]:
# Load match-level data that combines 2025 and 2026 seasons.
df = pd.read_csv("../data/processed/combined_processed_matches.csv")

# Ensure leak-free ordering for rolling feature creation.
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Quick preview of core columns.
df.head()

,gameid,date,patch,league,year,blue_team,red_team,blue_side_win
0,LOLTMNT03_179647,2025-01-11 11:11:24,15.01,LFL2,2025,IziDream,Team Valiant,0
1,LOLTMNT06_96134,2025-01-11 12:06:37,15.01,LFL2,2025,Esprit Shōnen,Skillcamp,1
2,LOLTMNT06_95160,2025-01-11 13:07:47,15.01,LFL2,2025,Karmine Corp Blue Stars,Project Conquerors,0
3,LOLTMNT03_178705,2025-01-11 14:03:27,15.01,LFL2,2025,Zerance,Lille Esport,0
4,LOLTMNT06_96169,2025-01-12 11:04:21,15.01,LFL2,2025,Team Valiant,Karmine Corp Blue Stars,1


## 3) Sanity-Check Dataset Composition

Inspect row counts, year distribution, and label balance to verify training input quality.

In [33]:
# Dataset size, seasonal composition, and target distribution.
print(df.shape)
print(df["year"].value_counts())
print(df["blue_side_win"].value_counts())

(13051, 8)
year
2025    9236
2026    3815
Name: count, dtype: int64
blue_side_win
1    6933
0    6118
Name: count, dtype: int64


## 4) Engineer Rolling Team Form Features

Create leak-free pre-match rolling features using each team's last 5 games, then store per-match differences for modeling.

In [34]:
# Number of recent matches used to compute rolling form.
window_size = 5

# Track each team's historical binary outcomes (1=win, 0=loss).
team_history = {}
feature_rows_5 = []

for _, row in df.iterrows():
    blue = row["blue_team"]
    red = row["red_team"]

    if blue not in team_history:
        team_history[blue] = []
    if red not in team_history:
        team_history[red] = []

    # Build pre-match rolling context from prior games only.
    blue_recent = team_history[blue][-window_size:]
    red_recent = team_history[red][-window_size:]

    blue_games_5 = len(blue_recent)
    red_games_5 = len(red_recent)

    blue_wr_5 = sum(blue_recent) / blue_games_5 if blue_games_5 > 0 else 0.5
    red_wr_5 = sum(red_recent) / red_games_5 if red_games_5 > 0 else 0.5

    feature_rows_5.append({
        **row,
        "blue_team_wr_5": blue_wr_5,
        "red_team_wr_5": red_wr_5,
        "blue_team_games_5": blue_games_5,
        "red_team_games_5": red_games_5,
        "wr_diff_5": blue_wr_5 - red_wr_5,
    })

    # Update team histories after observing the match outcome.
    if row["blue_side_win"] == 1:
        team_history[blue].append(1)
        team_history[red].append(0)
    else:
        team_history[blue].append(0)
        team_history[red].append(1)

rolling_df_5 = pd.DataFrame(feature_rows_5)
rolling_df_5.head()

,gameid,date,patch,league,year,blue_team,red_team,blue_side_win,blue_team_wr_5,red_team_wr_5,blue_team_games_5,red_team_games_5,wr_diff_5
0,LOLTMNT03_179647,2025-01-11 11:11:24,15.01,LFL2,2025,IziDream,Team Valiant,0,0.5,0.5,0,0,0.0
1,LOLTMNT06_96134,2025-01-11 12:06:37,15.01,LFL2,2025,Esprit Shōnen,Skillcamp,1,0.5,0.5,0,0,0.0
2,LOLTMNT06_95160,2025-01-11 13:07:47,15.01,LFL2,2025,Karmine Corp Blue Stars,Project Conquerors,0,0.5,0.5,0,0,0.0
3,LOLTMNT03_178705,2025-01-11 14:03:27,15.01,LFL2,2025,Zerance,Lille Esport,0,0.5,0.5,0,0,0.0
4,LOLTMNT06_96169,2025-01-12 11:04:21,15.01,LFL2,2025,Team Valiant,Karmine Corp Blue Stars,1,1.0,0.0,1,1,1.0


## 5) Validate Rolling Features

Check summary statistics and tail rows to confirm the engineered rolling signal is populated and sensible.

In [35]:
# Distribution of the rolling win-rate difference feature.
rolling_df_5[["wr_diff_5"]].describe()

# Recent rows to verify feature generation in later matches.
#rolling_df_5.tail()

,wr_diff_5
count,13051.000000
mean,-0.017079
std,0.378683
min,-1.000000
25%,-0.200000
50%,0.000000
75%,0.200000
max,1.000000


In [36]:
#Build rolling features with window = 10
window_size = 10

team_history = {}
feature_rows_10 = []

for _, row in df.iterrows():
    blue = row["blue_team"]
    red = row["red_team"]

    if blue not in team_history:
        team_history[blue] = []
    if red not in team_history:
        team_history[red] = []

    blue_recent = team_history[blue][-window_size:]
    red_recent = team_history[red][-window_size:]

    blue_games_10 = len(blue_recent)
    red_games_10 = len(red_recent)

    blue_wr_10 = sum(blue_recent) / blue_games_10 if blue_games_10 > 0 else 0.5
    red_wr_10 = sum(red_recent) / red_games_10 if red_games_10 > 0 else 0.5

    feature_rows_10.append({
        **row,
        "blue_team_wr_10": blue_wr_10,
        "red_team_wr_10": red_wr_10,
        "blue_team_games_10": blue_games_10,
        "red_team_games_10": red_games_10,
        "wr_diff_10": blue_wr_10 - red_wr_10,
    })

    if row["blue_side_win"] == 1:
        team_history[blue].append(1)
        team_history[red].append(0)
    else:
        team_history[blue].append(0)
        team_history[red].append(1)

rolling_df_10 = pd.DataFrame(feature_rows_10)
rolling_df_10.head()

,gameid,date,patch,league,year,blue_team,red_team,blue_side_win,blue_team_wr_10,red_team_wr_10,blue_team_games_10,red_team_games_10,wr_diff_10
0,LOLTMNT03_179647,2025-01-11 11:11:24,15.01,LFL2,2025,IziDream,Team Valiant,0,0.5,0.5,0,0,0.0
1,LOLTMNT06_96134,2025-01-11 12:06:37,15.01,LFL2,2025,Esprit Shōnen,Skillcamp,1,0.5,0.5,0,0,0.0
2,LOLTMNT06_95160,2025-01-11 13:07:47,15.01,LFL2,2025,Karmine Corp Blue Stars,Project Conquerors,0,0.5,0.5,0,0,0.0
3,LOLTMNT03_178705,2025-01-11 14:03:27,15.01,LFL2,2025,Zerance,Lille Esport,0,0.5,0.5,0,0,0.0
4,LOLTMNT06_96169,2025-01-12 11:04:21,15.01,LFL2,2025,Team Valiant,Karmine Corp Blue Stars,1,1.0,0.0,1,1,1.0


In [37]:
# Distribution of the rolling win-rate difference feature.
rolling_df_10[["wr_diff_10"]].describe()

# Recent rows to verify feature generation in later matches.
#rolling_df_10.tail()

,wr_diff_10
count,13051.000000
mean,-0.010102
std,0.310956
min,-1.000000
25%,-0.200000
50%,0.000000
75%,0.200000
max,1.000000


## 6) Train and Evaluate Rolling-Only Baseline

Fit logistic regression using only `wr_diff_5` to measure the isolated impact of short-term form.

In [38]:
# Rolling-only model (window = 5)

features_5 = ["wr_diff_5"]

X = rolling_df_5[features_5]
y = rolling_df_5["blue_side_win"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_5 = LogisticRegression(max_iter=1000)
model_5.fit(X_train, y_train)

y_pred_5 = model_5.predict(X_test)

acc_rolling_5 = accuracy_score(y_test, y_pred_5)

print("Rolling Only (5):", acc_rolling_5)

Rolling Only (5): 0.5928762926081961


In [39]:
# Rolling-only feature configuration.
features_10 = ["wr_diff_10"]

X = rolling_df_10[features_10]
y = rolling_df_10["blue_side_win"]

# Hold out test data for unbiased evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Rolling Only (10) Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Rolling Only (10) Accuracy: 0.6066641133665263
              precision    recall  f1-score   support

           0       0.61      0.54      0.57      1265
           1       0.61      0.67      0.64      1346

    accuracy                           0.61      2611
   macro avg       0.61      0.60      0.60      2611
weighted avg       0.61      0.61      0.60      2611



In [40]:
acc_rolling_10 = accuracy_score(y_test, y_pred)

## 7) Merge Rolling Feature Into All-Time Feature Table

Join rolling signal (`wr_diff_5`) with the existing combined all-time feature dataset keyed by `gameid`.

In [41]:
# Cell 10: Load combined all-time feature dataset
combined_all_time_df = pd.read_csv("../data/processed/combined_feature_matches.csv")
combined_all_time_df["date"] = pd.to_datetime(combined_all_time_df["date"])
combined_all_time_df.head()

,gameid,date,patch,league,year,blue_team,red_team,blue_side_win,date_year,blue_team_wr,red_team_wr,blue_team_games,red_team_games,wr_diff
0,LOLTMNT03_179647,2025-01-11 11:11:24,15.01,LFL2,2025,IziDream,Team Valiant,0,2025,0.5,0.5,0,0,0.0
1,LOLTMNT06_96134,2025-01-11 12:06:37,15.01,LFL2,2025,Esprit Shōnen,Skillcamp,1,2025,0.5,0.5,0,0,0.0
2,LOLTMNT06_95160,2025-01-11 13:07:47,15.01,LFL2,2025,Karmine Corp Blue Stars,Project Conquerors,0,2025,0.5,0.5,0,0,0.0
3,LOLTMNT03_178705,2025-01-11 14:03:27,15.01,LFL2,2025,Zerance,Lille Esport,0,2025,0.5,0.5,0,0,0.0
4,LOLTMNT06_96169,2025-01-12 11:04:21,15.01,LFL2,2025,Team Valiant,Karmine Corp Blue Stars,1,2025,1.0,0.0,1,1,1.0


In [42]:
# Merge rolling 10 into combined all-time features
combined_10_df = combined_all_time_df.merge(
    rolling_df_10[["gameid", "wr_diff_10"]],
    on="gameid",
    how="inner"
, validate="many_to_many")

combined_10_df.head()

,gameid,date,patch,league,year,blue_team,red_team,blue_side_win,date_year,blue_team_wr,red_team_wr,blue_team_games,red_team_games,wr_diff,wr_diff_10
0,LOLTMNT03_179647,2025-01-11 11:11:24,15.01,LFL2,2025,IziDream,Team Valiant,0,2025,0.5,0.5,0,0,0.0,0.0
1,LOLTMNT06_96134,2025-01-11 12:06:37,15.01,LFL2,2025,Esprit Shōnen,Skillcamp,1,2025,0.5,0.5,0,0,0.0,0.0
2,LOLTMNT06_95160,2025-01-11 13:07:47,15.01,LFL2,2025,Karmine Corp Blue Stars,Project Conquerors,0,2025,0.5,0.5,0,0,0.0,0.0
3,LOLTMNT03_178705,2025-01-11 14:03:27,15.01,LFL2,2025,Zerance,Lille Esport,0,2025,0.5,0.5,0,0,0.0,0.0
4,LOLTMNT06_96169,2025-01-12 11:04:21,15.01,LFL2,2025,Team Valiant,Karmine Corp Blue Stars,1,2025,1.0,0.0,1,1,1.0,1.0


## 8) Train and Evaluate Combined Feature Model

Fit a second logistic regression model using all-time form plus rolling form to test additive predictive value.

In [43]:
# Combined feature set: all-time + rolling signal.
features = [
    "wr_diff",
    "blue_team_games",
    "red_team_games",
    "wr_diff_10"
]

X = combined_10_df[features]
y = combined_10_df["blue_side_win"]

# Hold out test data for unbiased evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

acc_combined_10 = accuracy_score(y_test, y_pred)
print("Combined (10):", acc_combined_10)
print(classification_report(y_test, y_pred))

Combined (10): 0.6238988893144389
              precision    recall  f1-score   support

           0       0.64      0.51      0.57      1265
           1       0.61      0.73      0.67      1346

    accuracy                           0.62      2611
   macro avg       0.63      0.62      0.62      2611
weighted avg       0.63      0.62      0.62      2611



In [44]:
combined_5_df = combined_all_time_df.merge(
    rolling_df_5[["gameid", "wr_diff_5"]],
    on="gameid",
    how="inner",
    validate="one_to_one"
)

combined_5_df.to_csv("../data/processed/phase2_best_features.csv", index=False)

features_combined_5 = [
    "wr_diff",
    "blue_team_games",
    "red_team_games",
    "wr_diff_5"
]

X = combined_5_df[features_combined_5]
y = combined_5_df["blue_side_win"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

acc_combined_5 = accuracy_score(y_test, y_pred)

print("Combined (5):", acc_combined_5)

Combined (5): 0.6296438146304099


## 9) Track Experiment Outcomes

Store baseline and experiment slots in a single dictionary for easy comparison as additional window-size tests are completed.

In [45]:
# Centralized tracker for baseline and rolling-feature experiments.
results = {
    "Baseline (All-Time Only)": 0.6263,
    "Rolling Only (5)": acc_rolling_5,
    "Rolling Only (10)": acc_rolling_10,
    "Combined (5)": acc_combined_5,
    "Combined (10)": acc_combined_10
}

results

{'Baseline (All-Time Only)': 0.6263,
 'Rolling Only (5)': 0.5928762926081961,
 'Rolling Only (10)': 0.6066641133665263,
 'Combined (5)': 0.6296438146304099,
 'Combined (10)': 0.6238988893144389}